<a href="https://colab.research.google.com/github/Udaykiran606/Zepto-AI-ML-Capstone/blob/main/analysis/01_eda_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ==============================================================================
# MODULE 2: PART A — PROFILING, CLEANING & EDA (01_eda.ipynb)
# ==============================================================================

import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import StandardScaler

# Configure display options and visual aesthetics
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
sns.set_theme(style="whitegrid", palette="muted")

# ------------------------------------------------------------------------------
# TASK 1: DATA INGESTION, PROFILING & OFFLINE FALLBACK
# ------------------------------------------------------------------------------
print("=" * 70)
print("TASK 1: DATA INGESTION, PROFILING & FALLBACK ARTIFACT")
print("=" * 70)

os.makedirs("analytics", exist_ok=True)
csv_path = os.path.join("analytics", "titanic.csv")

# Load raw dataset once from Seaborn cache/network or offline fallback
try:
    df_raw = sns.load_dataset("titanic")
    df_raw.to_csv(csv_path, index=False)
    print(f"✅ Loaded raw dataset from Seaborn and saved fallback artifact: '{csv_path}'.")
except Exception:
    if os.path.exists(csv_path):
        df_raw = pd.read_csv(csv_path)
        print(f"✅ Loaded raw dataset from committed offline fallback: '{csv_path}'.")
    else:
        raise RuntimeError("Could not load dataset from network and no offline fallback found.")

print("\n--- Raw Dataset Profiling ---")
print(f"Shape: {df_raw.shape}")
print("\n--- Info ---")
df_raw.info()
print("\n--- Summary Statistics ---")
print(df_raw.describe(include="all"))

# Report missing value counts and percentages
missing_counts = df_raw.isnull().sum()
missing_pct = (missing_counts / len(df_raw)) * 100
affected = pd.DataFrame(
    {
        "Missing Count": missing_counts[missing_counts > 0],
        "Pct (%)": missing_pct[missing_pct > 0].round(2),
    }
).sort_values(by="Pct (%)", ascending=False)

print("\nAffected Missing Values (Pre-Cleaning):\n", affected)

# ------------------------------------------------------------------------------
# TASK 2: DEFENSIBLE DATA CLEANING & THRESHOLD RULES
# ------------------------------------------------------------------------------
print("\n" + "=" * 70)
print("TASK 2: DEFENSIBLE DATA CLEANING")
print("=" * 70)

df_clean = df_raw.copy()

# Rule A: High missingness (> 30%) -> Drop column (deck ~77.10% missing)
if "deck" in df_clean.columns:
    df_clean.drop(columns=["deck"], inplace=True)
    print("Rule A applied: Dropped column 'deck' (> 30% missing).")

# Rule B: 5%–30% missing range -> Grouped median imputation (age ~19.87% missing)
df_clean["age"] = df_clean.groupby(["pclass", "sex"])["age"].transform(
    lambda x: x.fillna(x.median())
)
print("Rule B applied: Imputed missing 'age' using grouped median (by pclass & sex).")

# Rule C: Under 5% missingness -> Drop rows (embarked / embark_town ~0.22% missing)
subset_cols = [c for c in ["embarked", "embark_town"] if c in df_clean.columns]
df_clean.dropna(subset=subset_cols, inplace=True)
print("Rule C applied: Dropped rows with missing 'embarked'/'embark_town' (< 5% missing).")

# Feature engineering (family size)
df_clean["family_size"] = df_clean["sibsp"] + df_clean["parch"] + 1

# Overwrite committed CSV with cleaned data artifact
df_clean.to_csv(csv_path, index=False)
print(f"\n✅ Cleaned dataset successfully overwritten to '{csv_path}'. Final Shape: {df_clean.shape}")

# ------------------------------------------------------------------------------
# TASK 3: UNIVARIATE ANALYSIS (IQR OUTLIERS & SKEWNESS)
# ------------------------------------------------------------------------------
print("\n" + "=" * 70)
print("TASK 3: UNIVARIATE ANALYSIS")
print("=" * 70)

def compute_iqr_outliers(series, name):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lb, ub = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outliers = series[(series < lb) | (series > ub)]
    print(f"{name} IQR Outliers: {len(outliers)} rows outside [{lb:.2f}, {ub:.2f}]")

compute_iqr_outliers(df_clean["age"], "Age")
compute_iqr_outliers(df_clean["fare"], "Fare")

fare_mean, fare_med, fare_mode = df_clean["fare"].mean(), df_clean["fare"].median(), df_clean["fare"].mode()[0]
print(f"\nFare Skewness check -> Mean: ${fare_mean:.2f} | Median: ${fare_med:.2f} | Mode: ${fare_mode:.2f}")
print("Written Interpretation: Since Mean ($32.20) > Median ($14.45) > Mode ($8.05), ticket fare is strongly RIGHT-SKEWED.")

# Histograms & Box Plots
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
sns.histplot(df_clean["age"], kde=True, ax=axes[0, 0], color="skyblue").set_title("Age Distribution")
sns.boxplot(x=df_clean["age"], ax=axes[0, 1], color="lightgreen").set_title("Age Box Plot")
sns.histplot(df_clean["fare"], kde=True, ax=axes[1, 0], color="salmon").set_title("Fare Distribution")
sns.boxplot(x=df_clean["fare"], ax=axes[1, 1], color="orchid").set_title("Fare Box Plot")
plt.tight_layout()
plt.savefig("analytics/univariate_plots.png")
plt.close()

# ------------------------------------------------------------------------------
# TASK 4: BIVARIATE ANALYSIS (BOOLEAN MASKING & 6x6 HEATMAP)
# ------------------------------------------------------------------------------
print("\n" + "=" * 70)
print("TASK 4: BIVARIATE ANALYSIS")
print("=" * 70)

# (a) Sex
for sex in df_clean["sex"].unique():
    print(f"Survival Rate - Sex ({sex}): {df_clean[df_clean['sex'] == sex]['survived'].mean():.2%}")

# (b) Pclass
for pclass in sorted(df_clean["pclass"].unique()):
    print(f"Survival Rate - Pclass ({pclass}): {df_clean[df_clean['pclass'] == pclass]['survived'].mean():.2%}")

# (c) Sex & Pclass
for sex in ["female", "male"]:
    for pclass in sorted(df_clean["pclass"].unique()):
        mask = (df_clean["sex"] == sex) & (df_clean["pclass"] == pclass)
        print(f"Survival Rate - Sex ({sex}) & Pclass ({pclass}): {df_clean[mask]['survived'].mean():.2%}")

# 6x6 Correlation Matrix (strictly specified numeric columns)
numeric_cols = ["survived", "pclass", "age", "sibsp", "parch", "fare"]
corr_matrix = df_clean[numeric_cols].corr()

# Programmatically extract top two off-diagonal correlation pairs
corr_unstacked = corr_matrix.abs().unstack()
off_diag = corr_unstacked[corr_unstacked.index.get_level_values(0) != corr_unstacked.index.get_level_values(1)]
top_pairs = off_diag.drop_duplicates().nlargest(2)

print("\n--- Top 2 Strongest Off-Diagonal Correlations ---")
for (f1, f2), val in top_pairs.items():
    raw_corr = corr_matrix.loc[f1, f2]
    print(f"Pair: ({f1}, {f2}) -> Pearson Correlation = {raw_corr:.4f} (Abs = {val:.4f})")

plt.figure(figsize=(7, 5))
sns.heatmap(corr_matrix, annot=True, fmt=".3f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("6x6 Correlation Matrix")
plt.tight_layout()
plt.savefig("analytics/correlation_heatmap.png")
plt.close()

# ------------------------------------------------------------------------------
# TASK 5: MULTIVARIATE DATA STORY & EDA STANDARDIZATION CHECK
# ------------------------------------------------------------------------------
print("\n" + "=" * 70)
print("TASK 5: MULTIVARIATE DATA STORY & EDA STANDARDIZATION CHECK")
print("=" * 70)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
sns.barplot(data=df_clean, x="pclass", y="survived", hue="sex", errorbar=None, ax=axes[0, 0]).set_title("1. Survival by Class & Sex")
sns.boxplot(data=df_clean, x="pclass", y="fare", hue="survived", ax=axes[0, 1]).set_yscale("log")
axes[0, 1].set_title("2. Log Fare by Class & Survival")
sns.scatterplot(data=df_clean, x="age", y="fare", hue="survived", style="sex", alpha=0.7, ax=axes[1, 0]).set_title("3. Age vs Fare by Survival & Sex")
sns.lineplot(data=df_clean, x="family_size", y="survived", hue="sex", marker="o", errorbar=None, ax=axes[1, 1]).set_title("4. Survival Rate by Family Size & Sex")
plt.tight_layout()
plt.savefig("analytics/multivariate_story.png")
plt.close()

# EDA Standardization Sanity Check (Z-score check)
eda_scaler = StandardScaler()
scaled_vals = eda_scaler.fit_transform(df_clean[["age", "fare"]])
print("EDA Standardization Check Summary:")
print(pd.DataFrame({
    "Feature": ["age", "fare"],
    "Raw Mean": [df_clean["age"].mean(), df_clean["fare"].mean()],
    "Scaled Mean": [scaled_vals[:, 0].mean(), scaled_vals[:, 1].mean()],
    "Raw Std": [df_clean["age"].std(), df_clean["fare"].std()],
    "Scaled Std": [scaled_vals[:, 0].std(), scaled_vals[:, 1].std()]
}))

TASK 1: DATA INGESTION, PROFILING & FALLBACK ARTIFACT
✅ Loaded raw dataset from Seaborn and saved fallback artifact: 'analytics/titanic.csv'.

--- Raw Dataset Profiling ---
Shape: (891, 15)

--- Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    object  
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    object  
 8   class        891 non-null    category
 9   who          891 non-null    object  
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    object  
 13  alive        891 non-null    object  
 14